In [25]:
# installing all the stuff we need upfront
!pip install langchain_community langchainhub chromadb langchain langchain-huggingface -q
!pip install transformers>=4.37.0 sentence-transformers huggingface_hub -q

In [26]:
from google.colab import userdata
import os

# pulling HF token from colab secrets
os.environ['HUGGINGFACEHUB_API_TOKEN'] = userdata.get('HuggingFace_API')

In [27]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0"
from langchain_community.document_loaders import WebBaseLoader

links = [
    "https://docs.flutter.dev/get-started/install",
    "https://docs.flutter.dev/get-started/codelab",
    "https://docs.flutter.dev/ui",
    "https://docs.flutter.dev/ui/widgets",
    "https://docs.flutter.dev/cookbook",
    "https://docs.flutter.dev/cookbook/navigation/navigation-basics",
    "https://docs.flutter.dev/cookbook/networking/fetch-data",
    "https://docs.flutter.dev/data-and-backend/state-mgmt/intro",
]

loader = WebBaseLoader(web_paths=links)
docs = loader.load()
print(f"loaded {len(docs)} pages")

loaded 8 pages


In [28]:
import re
from langchain_core.documents import Document

# raw scraped text has tons of junk — cleaning it up before chunking
clean_docs = []
for doc in docs:
    text = doc.page_content
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    text = text.strip()
    clean_docs.append(Document(page_content=text, metadata=doc.metadata))

print(f"cleaned {len(clean_docs)} docs")

cleaned 8 docs


In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

splits = text_splitter.split_documents(clean_docs)
print(f"got {len(splits)} chunks")

got 201 chunks


In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# embedding locally — no api calls, just runs on cpu
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("vector store ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vector store ready!


In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,          # greedy decoding — more focused answers
    return_full_text=False,   # only return the new generated part, not the prompt
)

llm = HuggingFacePipeline(pipeline=pipe)
print("model loaded — good to go!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


model loaded — good to go!


In [32]:
from langchain_core.prompts import PromptTemplate

# using a simple instruct-style format that TinyLlama actually follows well
prompt = PromptTemplate.from_template("""<|system|>
You are a Flutter documentation assistant. Answer ONLY using the context below. Be concise.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>""")

In [33]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("chain ready!")

chain ready!


In [34]:
response = rag_chain.invoke("What is Flutter and what can you build with it?")
print(response)

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Flutter is a cross-platform mobile development framework developed by Google that allows developers to create native mobile applications for iOS and Android devices. It provides a set of tools and libraries that simplify the development process, including a powerful Dart programming language, a Material Design UI library, and a Cupertino Design System for iOS. Flutter can be used to build a wide range of applications, including native mobile apps, web apps, and desktop applications. Some examples of Flutter-powered applications include the popular messaging app Signal, the popular music streaming service Spotify, and the popular e-commerce platform Shopify. Flutter is designed to be a flexible and scalable framework that can be used to build high-quality, user-friendly applications across multiple platforms.
